# RAG Chain

In [ ]:
from pathlib import Path

from langchain.chat_models import init_chat_model
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from loguru import logger

from chain_reaction.config import APIKeys
from chain_reaction.vector_store import load_vector_store

api_keys = APIKeys()

# Configure local data directory
path_parts = Path.cwd().parts
root_dir_index = path_parts.index("chain-reaction")
root_dir = Path(*path_parts[: root_dir_index + 1])
data_dir = root_dir / "data"
data_dir.mkdir(exist_ok=True)

## Load vector store

In [ ]:
vector_store = load_vector_store(
    collection_name="rag-wiki", persist_directory=data_dir / "chroma_db", api_key=api_keys.openai
)

# Number of documents in collection
num_documents = vector_store._client.get_collection("rag-wiki").count()
print(f"# Document chunks: {num_documents:>6,}")

## Retrieval subchain

In [ ]:
# Define a document retriever from the vector store
retriever = vector_store.as_retriever(
    search_type="mmr",  # Maximal Marginal Relevance (optimizes for relevance and diversity)
    search_kwargs={
        "k": 5,
        "fetch_k": 20,  # cast a wide net first
        "lambda_mult": 0.5,  # balance relevance vs diversity
    },
)


def get_context_docments(query: str) -> list[Document]:
    """Retrieve a list of relevant documents from the document store based on similarity to query string.

    Args:
        query (str): Query string to search for similar documents.

    Returns:
        list[Documents]: List of documents similar to query, ordered by relevance.
    """
    logger.info("Search for documents similar to query: {query}", query=query)
    return retriever.invoke(query)


def concat_documents(documents: list[Document], *, sep: str = "\n\n") -> str:
    """Concatenate a list of documents into a single block of text.

    Args:
        documents (list[Document]): Documents to concat.
        sep (str): Joining characters between documents.

    Returns:
        str: Combined text of documents.
    """
    logger.info("concatentating {num_docs} documents.", num_docs=len(documents))
    return sep.join(f"document: {doc.id}\n{doc.page_content}" for doc in documents)


retrieval_chain = RunnableLambda(get_context_docments) | RunnableLambda(concat_documents)

In [ ]:
# Invoke just the retrieval chain
print(retrieval_chain.invoke("What is a vector database?"))

## Rag Chain

In [ ]:
# Define a RAG prompt template
prompt_template = PromptTemplate.from_template(
    """You are a helpful question answering assistant.

    Instructions:
    -------------
    Use the following information (ordered by relevance): {context}

    To answer this question: {question}

    Critical Rules:
    ---------------
    - DO NOT make an answer up.
    - Use only information from the information provide to answer the question.
    - You MUST cite the document(s) that support your answer.
    """
)

# Chat model
chat_model = init_chat_model("gpt-5-nano", api_key=api_keys.openai, temperature=0.0)

# Full chain
rag_chain = (
    {"question": RunnablePassthrough(), "context": retrieval_chain} | prompt_template | chat_model | StrOutputParser()
)

## Invoke

In [ ]:
response: str = rag_chain.invoke("What is RAG?")
print(response)